# Programa de Integração Numérica em 2D
Este notebook resolve a seção 4.1 do [Programa 7](Programa_7.pdf), utilizando a função T(x, y) em uma malha n × n com as regras do Trapézio, Simpson 1/3 e Simpson 3/8.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Callable


In [3]:
def T(x: float, y: float) -> float:
    """Função de temperatura analítica conforme fornecida no enunciado."""
    return 2 * x * y + 2 * x - x**2 - 2 * y**2 + 72


In [4]:
def gerar_malha(n_x: int, n_y: int):
    """
    Gera a malha regular de n_x × n_y pontos no domínio [0,8] × [0,6]
    Retorna:
        x: vetor de coordenadas x
        y: vetor de coordenadas y
        df: DataFrame com colunas ['x', 'y', 'T']
    """
    x = np.linspace(0, 8, n_x)
    y = np.linspace(0, 6, n_y)
    X, Y = np.meshgrid(x, y)
    T_vals = T(X, Y)
    df = pd.DataFrame({'x': X.flatten(), 'y': Y.flatten(), 'T': T_vals.flatten()})
    return x, y, df


In [5]:
def trapezio_2d(f_vals: np.ndarray, dx: float, dy: float) -> float:
    """
    Integração 2D usando regra do trapézio.
    f_vals: matriz de valores da função T
    dx: passo em x
    dy: passo em y
    """
    ny, nx = f_vals.shape
    soma = f_vals[0, 0] + f_vals[0, -1] + f_vals[-1, 0] + f_vals[-1, -1]  # Canto
    soma += 2 * np.sum(f_vals[1:-1, 0]) + 2 * np.sum(f_vals[1:-1, -1])  # Bordas verticais
    soma += 2 * np.sum(f_vals[0, 1:-1]) + 2 * np.sum(f_vals[-1, 1:-1])  # Bordas horizontais
    soma += 4 * np.sum(f_vals[1:-1, 1:-1])                              # Interior
    return (dx * dy / 4) * soma


In [6]:
def simpson13_2d(f_vals: np.ndarray, dx: float, dy: float) -> float:
    """
    Integração 2D pela regra de Simpson 1/3.
    Requer número ímpar de pontos em cada direção.
    """
    ny, nx = f_vals.shape
    if nx % 2 == 0 or ny % 2 == 0:
        raise ValueError("Simpson 1/3 requer número ímpar de pontos em x e y.")
    def simpson_vec(vals, h):
        return h / 3 * (vals[0] + vals[-1] + 4 * np.sum(vals[1:-1:2]) + 2 * np.sum(vals[2:-2:2]))
    return simpson_vec(np.array([simpson_vec(row, dx) for row in f_vals]), dy)


In [7]:
def simpson38_2d(f_vals: np.ndarray, dx: float, dy: float) -> float:
    """
    Integração 2D pela regra de Simpson 3/8.
    Requer número de pontos múltiplo de 3 em cada direção.
    """
    ny, nx = f_vals.shape
    if (nx - 1) % 3 != 0 or (ny - 1) % 3 != 0:
        raise ValueError("Simpson 3/8 requer (n-1) múltiplo de 3 em x e y.")
    def simpson_vec(vals, h):
        return 3 * h / 8 * (vals[0] + vals[-1] + 3 * np.sum(vals[1:-1][(np.arange(1, len(vals)-1) % 3 != 0)]) + 2 * np.sum(vals[3:-3:3]))
    return simpson_vec(np.array([simpson_vec(row, dx) for row in f_vals]), dy)


In [8]:
def calcular_T_media_analitico() -> float:
    """Cálculo da média analítica da temperatura com integração simbólica."
    A integral dupla foi resolvida manualmente na questão como 2688.
    """
    return 2816 / (8 * 6)


In [9]:
ns = [3, 5, 7, 9, 13]  # Valores testados
t_verd = calcular_T_media_analitico()
A = 8*6

print(f"Temperatura real, obtida analiticamente: {t_verd:.4f}\n")
print(" n | Trapézio | Simpson 1/3 | Simpson 3/8")
print("------------------------------------------")
for n in ns:
    try:
        x, y, df = gerar_malha(n, n)
        dx = x[1] - x[0]
        dy = y[1] - y[0]
        f_vals = df['T'].values.reshape(n, n)

        t_trap = trapezio_2d(f_vals, dx, dy)
        t_simp13 = simpson13_2d(f_vals, dx, dy) if n % 2 == 1 else np.nan
        t_simp38 = simpson38_2d(f_vals, dx, dy) if (n-1) % 3 == 0 else np.nan

        print(f"{n:2} | {t_trap/A:8.4f} | {t_simp13/A:11.4f} | {t_simp38/A:11.4f}")
    except Exception as e:
        print(f"{n:2} | Erro: {str(e)}")


Temperatura real, obtida analiticamente: 58.6667

 n | Trapézio | Simpson 1/3 | Simpson 3/8
------------------------------------------
 3 |  53.0000 |     58.6667 |         nan
 5 |  57.2500 |     58.6667 |         nan
 7 |  58.0370 |     58.6667 |     58.6667
 9 |  58.3125 |     58.6667 |         nan
13 |  58.5093 |     58.6667 |     58.6667


# Conclusão

Apenas o método do trapézio não alcançou o a temperatura de acordo com o método analítico, já os métodos de Simpson chegaram no valor verdadeiro na primeira iteração válida em termos de número de elementos de cada algoritmo